# QKD BB84 Phase Control (Dual Channel + Auto-Cycle)

Control notebook for the QKD phase-encoded BB84 system on the RFSoC 4x2.

## Register Map

| Address | Name | R/W | Description |
|---------|------|-----|-------------|
| 0x00 | CTRL | R/W | `[0]` global_en, `[1]` alice_en, `[2]` bob_en, `[3]` auto_cycle_en |
| 0x04 | ALICE_PHASE_STAGED | R/W | `[1:0]` Staged Alice phase |
| 0x08 | BOB_PHASE_STAGED | R/W | `[1:0]` Staged Bob phase |
| 0x0C | STATUS | R | `[0]` alice_running, `[1]` bob_running, `[2]` auto_cycling, `[3]` sw_mode |
| 0x10 | PHASE_APPLY | R/W | Write 1 to latch both staged phases (auto-clears) |
| 0x14 | ACTIVE_PHASES | R | `[1:0]` alice active, `[3:2]` bob active |
| 0x18 | AUTO_FREQ_HZ | R | `[31:0]` Auto-cycle switching frequency in Hz |
| 0x1C | VERSION | R | 0x2026_0508 |

## Phase Encoding

| Value | Phase | Basis | Bit |
|-------|-------|-------|-----|
| 0b00 | 0 | Z | 0 |
| 0b01 | pi/2 | X | 0 |
| 0b10 | pi | Z | 1 |
| 0b11 | 3pi/2 | X | 1 |

## CTRL Shortcuts

| Value | Enables |
|-------|---------|
| 0x03 | Alice only |
| 0x05 | Bob only |
| 0x07 | Alice + Bob |
| 0x0B | Alice + auto-cycle |
| 0x0F | All |

## 1. Load Overlay and Initialize Hardware

In [ ]:
from pynq import PL
PL.reset()

import xrfdc
import xrfclk
from pynq import Overlay, MMIO, Clocks
import pprint

# Load the bitstream — update path to match your build output
xrfclk.set_ref_clks(lmk_freq=491.52, lmx_freq=491.52)
ol = Overlay('./qkd_phase_bb84.bit')

# Show all IP in the design
pprint.pprint(ol.ip_dict)

In [2]:
# Get handles to the QKD wrapper and RF data converter
# NOTE: update these names to match your block design instance names
# qkd = ol.qkd_top_wrapper_bd_0
qkd = ol.ip_dict['qkd_top_wrapper_bd_0']
base_addr = ol.ip_dict['qkd_top_wrapper_bd_0']['phys_addr']
addr_range = ol.ip_dict['qkd_top_wrapper_bd_0']['addr_range']
print(f"Base: {base_addr:#010x}, Range: {addr_range:#x}")
qkd_mmio = MMIO(base_addr, addr_range)
print(f"VERSION: {qkd_mmio.read(0x1C):#010x}")

rf = ol.usp_rf_data_converter_0
print(f"RF Data Converter: {rf}")

Base: 0x80000000, Range: 0x1000
VERSION: 0x20260425
RF Data Converter: <xrfdc.RFdc object at 0xffff7c152350>


## 2. Register Access Helpers

In [ ]:
# Register offsets
REG_CTRL                = 0x00
REG_ALICE_PHASE_STAGED  = 0x04
REG_BOB_PHASE_STAGED    = 0x08
REG_STATUS              = 0x0C
REG_PHASE_APPLY         = 0x10
REG_ACTIVE_PHASES       = 0x14
REG_AUTO_FREQ_HZ        = 0x18
REG_VERSION             = 0x1C

RFDC_CLK_FREQ_HZ = 307_200_000

PHASE_LABELS = {0: '0', 1: 'pi/2', 2: 'pi', 3: '3pi/2'}

def reg_read(offset):
    return qkd_mmio.read(offset)

def reg_write(offset, value):
    qkd_mmio.write(offset, value)

def set_phases(alice_phase, bob_phase):
    """Stage and atomically apply phase values for both channels."""
    reg_write(REG_ALICE_PHASE_STAGED, alice_phase & 0x3)
    reg_write(REG_BOB_PHASE_STAGED, bob_phase & 0x3)
    reg_write(REG_PHASE_APPLY, 1)

def set_alice_phase(phase):
    """Stage and apply Alice phase only."""
    reg_write(REG_ALICE_PHASE_STAGED, phase & 0x3)
    reg_write(REG_PHASE_APPLY, 1)

def set_bob_phase(phase):
    """Stage and apply Bob phase only."""
    reg_write(REG_BOB_PHASE_STAGED, phase & 0x3)
    reg_write(REG_PHASE_APPLY, 1)

def get_status():
    s = reg_read(REG_STATUS)
    return {
        'alice_running': bool(s & 0x1),
        'bob_running':   bool(s & 0x2),
        'auto_cycling':  bool(s & 0x4),
        'sw_mode':       bool(s & 0x8),
    }

def get_active_phases():
    a = reg_read(REG_ACTIVE_PHASES)
    return {
        'alice': PHASE_LABELS[a & 0x3],
        'bob':   PHASE_LABELS[(a >> 2) & 0x3],
    }

def get_auto_freq():
    return reg_read(REG_AUTO_FREQ_HZ)

def enable_auto_cycle():
    ctrl = reg_read(REG_CTRL)
    reg_write(REG_CTRL, ctrl | 0x08)

def disable_auto_cycle():
    ctrl = reg_read(REG_CTRL)
    reg_write(REG_CTRL, ctrl & ~0x08)

# Verify connectivity
version = reg_read(REG_VERSION)
print(f"VERSION: {version:#010x}")
assert version == 0x2026_0508, f"Unexpected version: {version:#010x}"

## 3. Configure RF Data Converter

Alice on DAC Tile 0 (100 MHz for AOD). Bob on DAC Tile 2 (90 MHz for AOM). Beat note = 10 MHz.

In [ ]:
# Alice — DAC Tile 0 (Tile 228)
alice_tile = rf.dac_tiles[0]
print(f"Alice Tile 0: PLL locked = {alice_tile.PLLLockStatus}")
alice_block = alice_tile.blocks[0]
alice_block.MixerSettings['Freq'] = 100.0
alice_block.UpdateEvent(xrfdc.EVENT_MIXER)
print(f"  Alice NCO: {alice_block.MixerSettings['Freq']} MHz")

# Bob — DAC Tile 2 (Tile 230)
bob_tile = rf.dac_tiles[2]
print(f"Bob Tile 2: PLL locked = {bob_tile.PLLLockStatus}")
bob_block = bob_tile.blocks[0]
bob_block.MixerSettings['Freq'] = 90.0
bob_block.UpdateEvent(xrfdc.EVENT_MIXER)
print(f"  Bob NCO: {bob_block.MixerSettings['Freq']} MHz")

## 4. Enable Outputs

CTRL shortcuts: 0x03=Alice only, 0x05=Bob only, 0x07=both, 0x0B=Alice+auto, 0x0F=all

In [ ]:
# Enable both channels
reg_write(REG_CTRL, 0x07)

status = get_status()
print(f"Status: {status}")
assert status['alice_running'], "Alice not running"
assert status['bob_running'], "Bob not running"

## 5. Set Phases

`set_phases(alice, bob)` for atomic update. Or `set_alice_phase()` / `set_bob_phase()` individually.

In [ ]:
# Set both phases
set_phases(alice_phase=0, bob_phase=0)
print(f"Active phases: {get_active_phases()}")

# Cycle Alice through all 4 phases while Bob stays at 0
import time
for p in range(4):
    set_alice_phase(p)
    time.sleep(0.1)
    phases = get_active_phases()
    print(f"Alice={phases['alice']:>5s}  Bob={phases['bob']:>5s}")

## 6. Auto-Cycle Mode (AOM Bandwidth Test)

Enable auto-cycle to have the PL automatically sweep through all 4 phases.
Use the board buttons to control switching rate:
- **btn[3]**: Cycle unit (Hz / KHz / MHz)
- **btn[2]**: Cycle increment (1 / 5 / 10)
- **btn[1]**: Increase frequency
- **btn[0]**: Decrease frequency

LED[3] lights up when auto-cycle is active. LED[1:0] shows current phase.

In [ ]:
# Enable auto-cycle on Alice (Bob stays at fixed phase for reference)
enable_auto_cycle()

status = get_status()
print(f"Status: {status}")
print(f"Auto-cycle freq: {get_auto_freq()} Hz")
print(f"Active phases: {get_active_phases()}")
print("\nUse buttons to adjust switching speed.")
print("APD210 should show 10 MHz beat note inverting at switch rate.")

## 7. Monitor Auto-Cycle (optional)

Poll and display the current switching frequency while you adjust with buttons.

In [ ]:
import time
from IPython.display import clear_output

try:
    for _ in range(30):  # monitor for 30 seconds
        freq = get_auto_freq()
        period = get_auto_period()
        phase = get_active_phase()
        clear_output(wait=True)
        print(f"Auto-cycle frequency: {freq} Hz")
        if period > 0:
            print(f"Period: {period} ticks ({RFDC_CLK_FREQ_HZ/period:.1f} Hz actual)")
        print(f"Current phase: {phase}")
        print(f"Status: {get_status()}")
        print("\n[Ctrl+C to stop monitoring]")
        time.sleep(1.0)
except KeyboardInterrupt:
    print("Monitoring stopped.")

In [ ]:
# Disable everything
disable_auto_cycle()
reg_write(REG_CTRL, 0x00)
print(f"Status: {get_status()}")